# Tyche calibration notebook

Worked example of using the `data/` downloads to calibrate the Tyche
simulation core. Run `data/scripts/fetch_all.sh && data/scripts/postprocess.sh`
first.

The notebook walks through:

1. Loading EBA Risk Dashboard NPL ratios.
2. Loading ECB MFI lending rates and volumes.
3. Loading FRED HY-OAS for stress-shock magnitude.
4. Re-fitting the `derive_pd_curve` constant in `tyche-sim` so simulated
   PDs match published EU NPL priors at the sector level.
5. Comparing simulator outputs against Moody's annual default study
   aggregates as a sanity check.

Requires: `pandas`, `openpyxl`, `numpy`, `matplotlib`, plus the
`tyche_sim` PyO3 binding built via `maturin develop --release` from
`crates/tyche-py`.

In [ ]:
from pathlib import Path
import json
import gzip
import pandas as pd

ROOT = Path('..').resolve()
RAW = ROOT / 'data' / 'raw'
PROC = ROOT / 'data' / 'processed'
print('raw dir:', RAW)
print('processed dir:', PROC)

## 1 — EBA NPL ratios

The EBA Risk Dashboard XLSX has many sheets. We're after the NPL-by-country
and NPL-by-sector tabs.

In [ ]:
eba_xlsx = next((RAW / 'eba').glob('risk_dashboard_*.xlsx'), None)
if eba_xlsx is None:
    print('No EBA XLSX found; rerun fetch_all.sh')
else:
    xls = pd.ExcelFile(eba_xlsx)
    print('sheets:', xls.sheet_names[:10], '…')

## 2 — FRED HY-OAS as the stress anchor

In [ ]:
hy = pd.read_csv(RAW / 'fred' / 'BAMLH0A0HYM2.csv', parse_dates=['observation_date']).rename(
    columns={'observation_date': 'date', 'BAMLH0A0HYM2': 'oas_pct'}
)
print(hy.tail())
p99 = hy['oas_pct'].quantile(0.99)
print(f'99th-percentile HY OAS (bps): {p99 * 100:.0f}')

## 3 — Re-fit the PD anchor in tyche-sim

Today `derive_pd_curve` uses `dd = ln(λ) / σ - 1.5`. The constant 1.5 is a
calibration anchor. Adjusting it shifts the entire PD curve.

Pseudocode:

```python
import tyche_sim
for c in [1.0, 1.25, 1.5, 1.75, 2.0]:
    # would require exposing the constant; see Phase 2
    pass
```

Phase 2 will expose this constant on `SimConfig` so the notebook can sweep
it without rebuilding the crate.